<a href="https://colab.research.google.com/github/Nipun-Pasindu/Statistical-Learning-E23210/blob/main/assignment%2004.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# 1. Environment Setup & Package Installation
# ==========================================
# Install the library directly from your GitHub repository including Plotly visualization dependencies
!pip install "git+https://github.com/Nipun-Pasindu/data-analysis-tool.git#egg=data-analysis-tool[plotting]"

DEPRECATION: git+https://github.com/Nipun-Pasindu/data-analysis-tool.git#egg=data-analysis-tool[plotting] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617
  Cloning https://github.com/Nipun-Pasindu/data-analysis-tool.git to /tmp/pip-install-q90wfjbn/data-analysis-tool_3357df53f89d491a9606daf498ec1494
  Running command git clone --filter=blob:none --quiet https://github.com/Nipun-Pasindu/data-analysis-tool.git /tmp/pip-install-q90wfjbn/data-analysis-tool_3357df53f89d491a9606daf498ec1494
  Resolved https://github.com/Nipun-Pasindu/data-analysis-tool.git to commit 5e8eb77dcd436d6f5b9a626ae999bef986a0410c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for data-analysis-tool: filename=data_analy

In [5]:
# ==========================================
# 2. Initialization & Mock Data Simulation
# ==========================================
import pandas as pd
import numpy as np
from data_analysis import DataInspector

# Instantiate the principal workspace manager
inspector = DataInspector()

# Generate a structural mixed dataset containing missing parameters, noise, and categorical factors
np.random.seed(42)
records_count = 120

mock_df = pd.DataFrame({
    'Age': np.random.choice([23, 45, 31, 60, np.nan, 28, 115], size=records_count), # Contains NaN and Outlier (115)
    'Salary': np.random.choice([52000, 98000, 71000, 125000, np.nan, 64000], size=records_count),
    'Experience': np.random.choice([1, 5, 8, 12, 20, np.nan], size=records_count),
    'Department': np.random.choice(['Engineering', 'HR', 'Marketing', 'Sales', ' '], size=records_count), # Contains empty space string
    'Gender': np.random.choice(['M', 'F', 'NULL', 'N/A'], size=records_count) # Contains typical textual null values
})

# Explicitly load into the inspector workspace (simulating the backend flow of inspector.upload_data())
# For interactive CSV uploads in Colab, uncomment and execute the line below:
# inspector.upload_data()

inspector.df = mock_df
print("Dataset successfully bound to workspace environment. Initial shape:", inspector.df.shape)

Dataset successfully bound to workspace environment. Initial shape: (120, 5)


In [6]:
# ==========================================
# 3. Exploration & Structural Cleaning Flow
# ==========================================
print("--- Initial Structural Analysis ---")
inspector.inspect_data()

print("\n--- Preprocessing: Missing Value Imputation ---")
# Impute using Pydantic validated strategy blocks
inspector.handle_missing_values(strategy='median')

print("\n--- Preprocessing: Duplicate Row Removal ---")
inspector.remove_duplicates()

print("\n--- Preprocessing: IQR Outlier Trimming ---")
# Clean numerical data arrays by trimming records extending past 1.5x IQR bounds
inspector.remove_outliers_iqr(columns=['Age', 'Salary'])

print("\n--- Final Cleaned Shape ---")
print(inspector.df.shape)

--- Initial Structural Analysis ---
=== Data Dimensions ===
Rows: 120 | Columns: 5

=== Column Types Summary ===
float64    3
object     2
Name: count, dtype: int64

Missing values global total: 56

=== Numerical Features Profile ===
            count          mean           std      min      25%      50%  \
Age         102.0     55.000000     31.609937     23.0     28.0     45.0   
Salary      105.0  78952.380952  27059.898286  52000.0  52000.0  71000.0   
Experience   97.0      8.061856      5.843113      1.0      1.0      8.0   

                75%       max  
Age            60.0     115.0  
Salary      98000.0  125000.0  
Experience     12.0      20.0  

=== Categorical Features Profile ===
           count unique top freq
Department   120      5       27
Gender       120      4   M   34

--- Preprocessing: Missing Value Imputation ---
Automated imputation pass executed using strategy parameter logic: 'median'.

--- Preprocessing: Duplicate Row Removal ---
Removed 5 exact duplicat

In [7]:
# ==========================================
# 4. Feature Engineering Pipelines
# ==========================================
print("--- Normalizing Numeric Features (Robust Scaling) ---")
numeric_scaled = inspector.extract_normalized_numeric_data(method='robust')
display(numeric_scaled.head())

print("\n--- Encoding Categorical Target Columns (One-Hot) ---")
categorical_encoded = inspector.extract_normalized_categorical_data(method='onehot')
display(categorical_encoded.head())

print("\n--- Assembling Unified Model-Ingestion Matrix ---")
final_training_df = inspector.create_normalized_data_df(numeric_method='robust', categorical_method='onehot')
display(final_training_df.head())

--- Normalizing Numeric Features (Robust Scaling) ---


,Age,Salary,Experience
0,0.722892,-0.558824,-1.000000
1,0.000000,-0.558824,-1.000000
2,-0.674699,1.588235,0.000000
3,0.000000,0.000000,-0.428571
4,0.000000,0.000000,1.714286



--- Encoding Categorical Target Columns (One-Hot) ---


,Department_Engineering,Department_HR,Department_Marketing,Department_Sales,Gender_M,Gender_N/A,Gender_NULL
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0



--- Assembling Unified Model-Ingestion Matrix ---


,Age,Salary,Experience,Department_Engineering,Department_HR,Department_Marketing,Department_Sales,Gender_M,Gender_N/A,Gender_NULL
0,0.722892,-0.558824,-1.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,-0.558824,-1.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.674699,1.588235,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.000000,0.000000,-0.428571,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,0.000000,0.000000,1.714286,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [8]:
# ==========================================
# 5. Interactive Visual Exploration
# ==========================================
# Render the 3-pane structural diagnostic visualization matrix (Violin, Scatter, Histogram)
inspector.plot_numerical(columns=['Age', 'Salary'])

In [9]:
# ==========================================
# 6. Context-Aware Bivariate Layouts
# ==========================================
print("--- Numerical vs Numerical relationship ---")
inspector.plot_relationship(col1='Age', col2='Salary')

print("\n--- Categorical vs Numerical relationship ---")
inspector.plot_relationship(col1='Department', col2='Salary')

--- Numerical vs Numerical relationship ---



--- Categorical vs Numerical relationship ---


In [10]:
# ==========================================
# 7. Advanced Statistical Mapping & Heatmaps
# ==========================================
# Generate an all-inclusive mixed-type heatmap combining Pearson, Cramér's V, and Eta coefficients
inspector.plot_all_associations_heatmap()

In [11]:
# ==========================================
# 8. Isolated Direct Custom Plotting Calls
# ==========================================
# Access low-level programmatic plots yielding encapsulated dictionary payloads
bar_packet = inspector.plot_bar_chart(
    x='Department',
    y='Salary',
    color='Gender',
    barmode='group',
    data=inspector.df
)

# Output rendering step
inspector.display_image(bar_packet)

In [12]:
# ==========================================
# 9. Low-level Custom Distribution Binning
# ==========================================
hist_packet = inspector.plot_histogram(
    x='Age',
    bins=[0, 25, 40, 60, 100],
    title='Demographics Grouped Breakdown Profiles',
    data=inspector.df
)

inspector.display_image(hist_packet)